In [ ]:
import pandas as pd


# 1. События взаимодействия с оффером
offer_events = (
    mobapp_act[mobapp_act['metric_id'] == 1]
    [['magnit_id', 'calc_date']]
    .drop_duplicates()
    .copy()
)

offer_events['calc_date'] = pd.to_datetime(offer_events['calc_date'])


# 2. Статусы подписки
subscr = subscr_status.copy()

subscr['subscr_date_act'] = pd.to_datetime(
    subscr['subscr_date_act']
)


# 3. Мержим события с подписками
events_subscr = offer_events.merge(
    subscr[['magnit_id', 'subscr_date_act', 'applied']],
    on='magnit_id',
    how='left'
)


# 4. Флаг: нет активной подписки после коммуникации
events_subscr['without_active_subscr_after_event'] = (
    events_subscr['subscr_date_act'].isna()
    |
    (
        (events_subscr['subscr_date_act'] >= events_subscr['calc_date'])
        & (events_subscr['applied'] == 1)
    )
)


# 5. Клиенты, у которых был хотя бы один просмотр/клик без активной подписки после него
clients_without_subscr = (
    events_subscr[
        events_subscr['without_active_subscr_after_event']
    ][['magnit_id']]
    .drop_duplicates()
)

clients_without_subscr['has_event_without_subscr'] = 1


# 6. Мержим с когортами
result = client_cohorts.merge(
    clients_without_subscr,
    on='magnit_id',
    how='left'
)

result['has_event_without_subscr'] = (
    result['has_event_without_subscr']
    .fillna(0)
)


# 7. Метрики по когортам
cohort_result = (
    result
    .groupby('campaigns_cnt', as_index=False)
    .agg(
        client_cnt=('client_id', 'nunique'),
        event_without_subscr_cnt=('has_event_without_subscr', 'sum')
    )
)

cohort_result['event_without_subscr_pct'] = (
    cohort_result['event_without_subscr_cnt']
    / cohort_result['client_cnt']
    * 100
).round(2)

cohort_result